In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# General

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame
from listUtils import getFlatList
from musicdb import MusicDBIO
from sys import prefix
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

MasterPaths()
  ==> Raw:        /Volumes/Piggy/Discog
  ==> Mod:        /Volumes/Seagate/Discog
  ==> Sum:        /Users/tgadfort/Music/Discog
MasterMetas()
  ==> Media:      ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Metas:      ['Basic', 'Media', 'Genre', 'Bio', 'Link', 'Metric', 'Counts']
  ==> Searches:   ['Name', 'AlbumMedia', 'SingleEPMedia']
MasterDBs()
  ==> DBs:        ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM']
MasterParams()
  ==> ModVals:    100
  ==> Project:    pandb
  ==> MusicDB:    musicdb


# DB-Specific

In [3]:
from lib import metalarchives
mio   = metalarchives.MusicDBIO(verbose=False, mkDirs=False)
apiio = metalarchives.RawAPIData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant MetalArchives DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/MetalArchives


# Master Files

In [ ]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
localArtists       = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
masterArtistsData  = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtistsData".format(db.lower()))
searchArtists      = mio.data.getSearchArtistData()
knownArtists       = mio.data.getSummaryNameData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [ ]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Local Master Search:       {0}".format(len(localArtists.get())))
print("   Global Master Search:      {0}".format(len(masterArtists.get())))
print("   Global Master Search Data: {0}".format(len(masterArtistsData.get())))
print("   Search Artists:            {0}".format(len(searchArtists)))
print("   Errors:                    {0}".format(len(errors.get())))
print("   Known Summary IDs:         {0}".format(len(knownArtists)))

# Search For New Artists

In [ ]:
mio   = metalarchives.MusicDBIO(verbose=False,local=True,mkDirs=False)
apiio = metalarchives.RawAPIData(debug=False)

In [ ]:
useStarterData  = False
useMasterData   = False
useRYMData      = True
useSpotifyData  = False
useAllMusicData = False # Last one done
useLastFMData   = False
useAOTYData     = False

if useStarterData is True:
    starterData = io.get("starter.p")
    artistNames = Series({v["Ref"].split("/")[-1]: v["Name"] for k,v in starterData.items()})
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
elif useSpotifyData is True:
    from gate import MusicDBGate
    mdbgate = MusicDBGate()
    spotio = mdbgate.getIO(db="Spotify")
    spotGenreData = DataFrame(spotio.data.getSummaryNameData()).join(spotio.data.getSummaryGenreData())
    def isMetal(genres):
        if isinstance(genres,list):
            tests = ["Metal", "Black", "Death", "Thrash", "Doom", "Hard"]
            for test in tests:
                for genre in genres:
                    if genre.title().find(test) != -1:
                        return True
            return False

    spotGenreData = spotGenreData[spotGenreData["Genre"].notna()]
    artistNames = spotGenreData[spotGenreData["Genre"].apply(isMetal)]["Name"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]   

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet))) 
elif useAllMusicData is True:
    from gate import MusicDBGate
    mdbgate = MusicDBGate()
    amio = mdbgate.getIO(db="AllMusic")
    amGenreData = DataFrame(amio.data.getSummaryNameData()).join(amio.data.getSummaryGenreData())
    def isMetal(genres):
        if isinstance(genres,list):
            tests = ["Metal", "Black", "Death", "Thrash", "Doom", "Hard"]
            for test in tests:
                for genre in genres:
                    if genre.title().find(test) != -1:
                        return True
            return False

    amGenreData = amGenreData[amGenreData["Tag"].notna()]
    artistNames = amGenreData[amGenreData["Tag"].apply(isMetal)]["Name"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]   

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet))) 
elif useRYMData is True:
    from gate import MusicDBGate
    mdbgate = MusicDBGate()
    rymio = mdbgate.getIO(db="RateYourMusic")
    rymGenreData = DataFrame(rymio.data.getSummaryNameData()).join(rymio.data.getSummaryGenreData())
    def isMetal(genres):
        if isinstance(genres,list):
            tests = ["Metal", "Black", "Death", "Thrash", "Doom"]
            for test in tests:
                for genre in genres:
                    if genre.find(test) != -1:
                        return True
            return False

    rymGenreData = rymGenreData[rymGenreData["Genre"].notna()]
    artistNames = rymGenreData[rymGenreData["Genre"].apply(isMetal)]["Name"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
elif useMasterData is True:
    from musicdb import MusicDBIO
    pdbio = MusicDBIO()
    mmeDF = pdbio.getData()
    artistNames       = mmeDF[mmeDF["AllMusic"].isna()]["ArtistName"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    

#   Artist Names To Get:  8443
#   Artist Names To Get:  4486

## Download Artist Searches

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "6:50am")
#tt = TermTime("tomorrow", "11:00am")
tt = TermTime("today", "9:00pm")
maxN = 50000000

n  = 0
masterArtistsDict     = masterArtists.get()
masterArtistsDataDict = masterArtistsData.get()
searchedForErrors     = errors.get()
for i,(idx,artistName) in enumerate(artistNamesToGet.iteritems()):
    if masterArtistsDict.get(artistName) is not None:
        continue

    response = apiio.getArtistSearchResults(artistName=artistName)
    if response is None:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistName] = True
        apiio.sleep(3.5)
    
    masterArtistsDict[artistName]     = True
    masterArtistsDataDict[artistName] = response
    apiio.sleep(3.5)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
        masterArtists.save(data=masterArtistsDict)
        masterArtistsData.save(data=masterArtistsDataDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

if True:
    ts.stop()
    print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
    masterArtists.save(data=masterArtistsDict)
    masterArtistsData.save(data=masterArtistsDataDict)
    if len(searchedForErrors) > 0:
        errors.save(data=searchedForErrors)

## Save Results

In [ ]:
from pandas import concat
df = masterArtistsData.get()
if isinstance(df,dict):
    print("Found {0} New Artists".format(len(df)))
    searchDF = metalarchives.MusicDBIO(local=False).data.getSearchArtistData()
    if isinstance(searchDF,Series):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF, Series(df)])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Results".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Results".format(searchDF.shape[0]))
    artists = {}
    for artistName,artistResults in searchDF.iteritems():
        if artistResults is not None:
            for item in artistResults:
                artists[item['id']] = item['name']
    print("Found {0} Unique Artists".format(len(artists)))
    s = Series(artists)
    print("  ==> {0} Old Artists".format(len(s[s.index.isin(knownArtists.index)])))
    print("  ==> {0} New Artists".format(len(s[~s.index.isin(knownArtists.index)])))
    print("Saving Data")
    metalarchives.MusicDBIO(local=False).data.saveSearchArtistData(data=searchDF)

In [ ]:
masterArtistsData.save(data={})

# Download Artist Data

In [ ]:
mio   = metalarchives.MusicDBIO(verbose=False,local=True,mkDirs=False)
apiio = metalarchives.RawAPIData(debug=False)

## Find Artists To Download

In [ ]:
artistData = {}
for searchTerm,searchResults in searchArtists.iteritems():
    if isinstance(searchResults,list):
        artistData.update({item["id"]: item for item in searchResults if isinstance(item,dict)})
artistData       = DataFrame(artistData).T.sort_values(by='id')
artistNames      = artistData[["name", "url"]]
localArtistsDict = localArtists.get()
artistIDsToGet   = artistNames[~artistNames.index.isin(localArtistsDict.keys())].sample(frac=1)

print("{0} Search Results".format(db))
print("   Available IDs:      {0}".format(len(artistNames)))
print("   Known Artist IDs:   {0}".format(len(localArtistsDict)))
print("   Artist IDs To Get:  {0}".format(len(artistIDsToGet)))

#   Artist IDs To Get:  1720

## Download The Data

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} Artist Data".format(db))
#tt = TermTime("tomorrow", "11:00am")
tt = TermTime("today", "7:00pm")
maxN = 50000000

n  = 0
localArtistsDict     = localArtists.get()
searchedForErrors    = errors.get()
for i,(artistID,row) in enumerate(artistIDsToGet.iterrows()):
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue

    artistName = row["name"]
    artistURL  = row["url"]

    dbID   = artistID
    modVal = mio.mv.get(dbID)
    if mio.data.getRawArtistInfoFilename(modVal, dbID).exists():
        localArtistsDict[artistID] = artistName
        continue
    try:
        response = apiio.getArtistInfoResults(artistName=artistName, artistURL=artistURL)
    except:
        response = None
    if response is None:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistID] = True
        apiio.sleep(3.5)
        continue
    
    localArtistsDict[artistID] = artistName
    mio.data.saveRawArtistInfoData(data=response, modval=modVal, dbID=dbID)
    apiio.sleep(5.5)
    n += 1
        
    if n % 5 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

ts.stop()
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

In [ ]:
del searchedForErrors['118073']
errors.save(data=searchedForErrors)


In [4]:
metalarchives.moveLocalFiles()

MusicDBBaseDirs(db=MetalArchives)
   Using Local Path For Raw Data <<====
   RawDataDir     = /Users/tgadfort/Music/Discog/artists-metalarchives
   ModValDataDir  = /Volumes/Seagate/Discog/artists-metalarchives-db
   MetaDataDir    = /Volumes/Seagate/Discog/artists-metalarchives-db/metadata
   SummaryDataDir = /Users/tgadfort/Music/Discog/db-metalarchives
MetalArchives SummaryData
  ==> Basic
  ==> Media
  ==> Genre
  ==> Bio
  ==> Link
  ==> Metric
ParseRawDataUtils(mdbdata, mdbdir) [MetalArchives]
MetalArchives ModValMetaData
  ==> Basic (Universal)
  ==> Media (Media)
  ==> Genre
  ==> Bio
  ==> Link
MusicDBBaseDirs(db=MetalArchives)
   RawDataDir     = /Volumes/Piggy/Discog/artists-metalarchives
   ModValDataDir  = /Volumes/Seagate/Discog/artists-metalarchives-db
   MetaDataDir    = /Volumes/Seagate/Discog/artists-metalarchives-db/metadata
   SummaryDataDir = /Users/tgadfort/Music/Discog/db-metalarchives
MetalArchives SummaryData
  ==> Basic
  ==> Media
  ==> Genre
  ==> Bio
  ==> 

Running glob(/Users/tgadfort/Music/Discog/artists-metalarchives/8/*.p)
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/8/3540474008.p to /Volumes/Piggy/Discog/artists-metalarchives/8/3540474008.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/8/3540390208.p to /Volumes/Piggy/Discog/artists-metalarchives/8/3540390208.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/8/3540431408.p to /Volumes/Piggy/Discog/artists-metalarchives/8/3540431408.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/8/3540295408.p to /Volumes/Piggy/Discog/artists-metalarchives/8/3540295408.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/8/14408.p to /Volumes/Piggy/Discog/artists-metalarchives/8/14408.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/8/34508.p to /Volumes/Piggy/Discog/artists-metalarchives/8/34508.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/8/3540322708.p to /Volumes/Piggy/Discog

  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/14/68714.p to /Volumes/Piggy/Discog/artists-metalarchives/14/68714.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/14/3540380314.p to /Volumes/Piggy/Discog/artists-metalarchives/14/3540380314.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/14/3540377514.p to /Volumes/Piggy/Discog/artists-metalarchives/14/3540377514.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/14/3540439514.p to /Volumes/Piggy/Discog/artists-metalarchives/14/3540439514.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/14/3540351414.p to /Volumes/Piggy/Discog/artists-metalarchives/14/3540351414.p
Running glob(/Users/tgadfort/Music/Discog/artists-metalarchives/15/*.p)
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/15/3540274215.p to /Volumes/Piggy/Discog/artists-metalarchives/15/3540274215.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/15/3540428615.p 

  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/23/22123.p to /Volumes/Piggy/Discog/artists-metalarchives/23/22123.p
Running glob(/Users/tgadfort/Music/Discog/artists-metalarchives/24/*.p)
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/24/114424.p to /Volumes/Piggy/Discog/artists-metalarchives/24/114424.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/24/3540366624.p to /Volumes/Piggy/Discog/artists-metalarchives/24/3540366624.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/24/60024.p to /Volumes/Piggy/Discog/artists-metalarchives/24/60024.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/24/3540434524.p to /Volumes/Piggy/Discog/artists-metalarchives/24/3540434524.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/24/3540456224.p to /Volumes/Piggy/Discog/artists-metalarchives/24/3540456224.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/24/31824.p to /Volumes/Piggy/Disco

Running glob(/Users/tgadfort/Music/Discog/artists-metalarchives/33/*.p)
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/33/3540358833.p to /Volumes/Piggy/Discog/artists-metalarchives/33/3540358833.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/33/11733.p to /Volumes/Piggy/Discog/artists-metalarchives/33/11733.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/33/3540389433.p to /Volumes/Piggy/Discog/artists-metalarchives/33/3540389433.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/33/115233.p to /Volumes/Piggy/Discog/artists-metalarchives/33/115233.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/33/3540488733.p to /Volumes/Piggy/Discog/artists-metalarchives/33/3540488733.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/33/3540373733.p to /Volumes/Piggy/Discog/artists-metalarchives/33/3540373733.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/33/3540278833.p to /Volu

...

  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/61/5261.p to /Volumes/Piggy/Discog/artists-metalarchives/61/5261.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/61/45861.p to /Volumes/Piggy/Discog/artists-metalarchives/61/45861.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/61/3540279061.p to /Volumes/Piggy/Discog/artists-metalarchives/61/3540279061.p
Running glob(/Users/tgadfort/Music/Discog/artists-metalarchives/62/*.p)
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/62/3540286862.p to /Volumes/Piggy/Discog/artists-metalarchives/62/3540286862.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/62/8362.p to /Volumes/Piggy/Discog/artists-metalarchives/62/8362.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/62/3540255762.p to /Volumes/Piggy/Discog/artists-metalarchives/62/3540255762.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/62/3540486762.p to /Volumes/Piggy/Discog

  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/71/3540390571.p to /Volumes/Piggy/Discog/artists-metalarchives/71/3540390571.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/71/3540294071.p to /Volumes/Piggy/Discog/artists-metalarchives/71/3540294071.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/71/3540473871.p to /Volumes/Piggy/Discog/artists-metalarchives/71/3540473871.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/71/87071.p to /Volumes/Piggy/Discog/artists-metalarchives/71/87071.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/71/3540419871.p to /Volumes/Piggy/Discog/artists-metalarchives/71/3540419871.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/71/3540437671.p to /Volumes/Piggy/Discog/artists-metalarchives/71/3540437671.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/71/3540413371.p to /Volumes/Piggy/Discog/artists-metalarchives/71/3540413371.p
  ==> Mov

Running glob(/Users/tgadfort/Music/Discog/artists-metalarchives/79/*.p)
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/79/67679.p to /Volumes/Piggy/Discog/artists-metalarchives/79/67679.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/79/82679.p to /Volumes/Piggy/Discog/artists-metalarchives/79/82679.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/79/7579.p to /Volumes/Piggy/Discog/artists-metalarchives/79/7579.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/79/4579.p to /Volumes/Piggy/Discog/artists-metalarchives/79/4579.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/79/3540476279.p to /Volumes/Piggy/Discog/artists-metalarchives/79/3540476279.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/79/13179.p to /Volumes/Piggy/Discog/artists-metalarchives/79/13179.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/79/72079.p to /Volumes/Piggy/Discog/artists-metalarchives/79

  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/89/29089.p to /Volumes/Piggy/Discog/artists-metalarchives/89/29089.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/89/3540319189.p to /Volumes/Piggy/Discog/artists-metalarchives/89/3540319189.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/89/128889.p to /Volumes/Piggy/Discog/artists-metalarchives/89/128889.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/89/14789.p to /Volumes/Piggy/Discog/artists-metalarchives/89/14789.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/89/35189.p to /Volumes/Piggy/Discog/artists-metalarchives/89/35189.p
Running glob(/Users/tgadfort/Music/Discog/artists-metalarchives/90/*.p)
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/90/3540503990.p to /Volumes/Piggy/Discog/artists-metalarchives/90/3540503990.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/90/3540500790.p to /Volumes/Piggy/Discog/art

  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/98/66998.p to /Volumes/Piggy/Discog/artists-metalarchives/98/66998.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/98/87398.p to /Volumes/Piggy/Discog/artists-metalarchives/98/87398.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/98/3540436798.p to /Volumes/Piggy/Discog/artists-metalarchives/98/3540436798.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/98/3540397098.p to /Volumes/Piggy/Discog/artists-metalarchives/98/3540397098.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/98/3540442298.p to /Volumes/Piggy/Discog/artists-metalarchives/98/3540442298.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/98/3540268198.p to /Volumes/Piggy/Discog/artists-metalarchives/98/3540268198.p
  ==> Moving /Users/tgadfort/Music/Discog/artists-metalarchives/98/84198.p to /Volumes/Piggy/Discog/artists-metalarchives/98/84198.p
  ==> Moving /Users/tgadfort/